In [1]:
#Project Name: Transfer Learning with CNN to Perform Object Identification 
#Project Author: Kent Roller
#Project Class: IST 691 Applied Deep Learning 
#Project Creation Date: May 5th, 2025


#Project Purpose: To leverage transfer learning through models such as YOLOv5 
# to perform object identification. The overall goal is two fold with the project
#and as such the project will be completed in two stages. The first stage starts
# with getting prerequisites installed and operational such as pytorch. 
# Next will setup a model pipeline to be called for new inputs. The idea here
# is that the pipeline will run and determine if there is a vehicle within a 
# given frame. If a vehicle is present then the next model will run which will
# either identify the license plate and extract the text or move straight to 
# extracting the text if possible. The completion of the first stage is accomplished
# when the model can be used with video input provided by the user. 
# The second stage will also leverage transfer learning but is dependent on 
# the completion of the first stage. The object detection for the second stage 
# will attempt to a.) recognize the drivers area and the driver of the vehicle
# and b.) will be able to recognize if there is an object in the drivers hand 
#such as a phone, tablet, book, etc. 
#If success is found in both the first and second stage attempts will be made 
# at a rudimentary storage solution for objects identified to catalouge the 
# license plate and driver shot together. Further attempts will then be made 
# to incorporate geo-tag information from the camera but this will require the 
# purchase of such a camera and additional work to incorporate a live feed. 
# The end goal is to create a system that can in real time analyze 1-2 frames
# from a 60fps camera feed in real time and perform the model pipeline 
# automatically, with the byproduct being a project that could feasibly be rolled
# into production at some point. 


#NOTE!!!!!!
#ADDITION JUNE 18 2025
#After scrapping some trash pipes in the code and rewritting them from scratch finally happy with overall performance
#please bear in mind that I am aware the implemnentation here is sloppy, I plan to redo several aspects: 
#The code will be rewritten in its entirety in VSCode instead of Jupyter 
#This will allow for much cleaner implementation of the individual pipes instead of janky cell magic use 
#This will also allow for a requirements.txt file which will make production implementation much simplier
#Once these aspects are handled will begin additions to handle video input instead of still photos

#other notes:
#Think at this point the system would work best with two cameras, one facing forward and the other facing
#perpendicular to it to aim into the cabin of a vehicle. If the first model is a success it will then take input 
#from the second camera and intialize the second stage. Otherwise would need to develop a way to rotate the camera 
#per vehicle and given the cost of cameras they are cheap enough to where two camera setups should be tenable.
#Again, the point of this is to mount this system into a vehicle and allow for patrol with minimal interaction with 
#the system, so manual rotation of the camera or fiddling with the program while driving is not an option
#Then need to work out some kind of storage solution i.e. database to store output in 
#then may look at hosting in container via docker to allow for remote/non-local execution

#All in all pretty happy with whats been accomplished here so far, and hope to flesh it out by the end of the summer



In [2]:
#Some notes about the enviroment that the code is ran in: 
#Python version: 3.9.21
#Conda version: 25.6
#Pytorch Version: 3.9
#Torch: 2.5.1
#CUDA Version: 12.7
#Nvidia Driver: 566.14
#IDE: Jupyter Lab
#Collab was not used due to aggresive timeout in work enviroment
#ultraytics version: 8.3.130
#mediapipe version: 0.10.8
#YoloV8 -- object identification

#Prerequisites such as torch, yolov5, and easyocr are installed via conda terminal not in IDE. 
#Additional Notes: 
#Initially planned to use huggingface_hub to access trained weights from license plate model
#Found that HuggingFace_hub versions 0.25.1 and up are not compatible with YOLOv5 7.0.3
#Downgraded huggingface_hub to 0.24.1 and resolved errors, however connection issues still present with 
# huggingface_hub which are assumed at this point to be network related. 
#Changed to using git version 2.49.0.windows.1 and using same weights found in previous target 
# The repo cloned for the license plate weights is:
# wasdac9/automatic-number-plate-recognition
# Weights from orignial huggingface model target are no longer hosted on git or github...

In [14]:
#Import torch for use 
import torch

In [15]:
#Write the cell block to disc so it can be called down below as an imported function
#%%writefile vehicle_detector.py
#Cant use magic cell functions with pytorch, will have to try this another way
from IPython import get_ipython
code1=r"""
import torch
#Load Yolo in via torch hub one time during application start
model_vehicle = torch.hub.load(
    'ultralytics/yolov5',
    'yolov5s',
    #Here we want to be sure to specify the pretrained model as 
    #we want to make use of the weights already created on previous datasets
    pretrained=True
)

#Need to specify which classes we want to use for object identification in the initial pass
# For this we will be using COCO instead of IMAGENET data with 4 classes to start with. 
# If model performance is not satisfactory will adjust classes as required.
# Here, 2=Car, 3=Motorcycle, 5=Bus, and 7=Truck
Vehicle_Classes = {2,3,5,7}

#Create function to detect vehicle in an image utilizing YOLO
def detect_vehicle(img):
    #Returns True and Bounding boxes for image if vehicle exists in given frame
    results = model_vehicle(img, size=640, augment=False)
    #Here the code isnt descriptive but xyxy gives the output for x1,y1,x2,y2,conf, and class
    detections=results.xyxy[0]
    #Bounding box layer for model if vehicle is detected
    veh_boxes = [d[:4].cpu().numpy() for d in detections if int (d[5]) in Vehicle_Classes]
    return len(veh_boxes) > 0, veh_boxes
"""
#Write the code1 chunk to function using imported ipyhton package
get_ipython().run_cell_magic('writefile', 'vehicle_detector.py', code1)
#Ensure the file was written as expected 
print(" vehicle_detector.py written({} bytes)".format(len(code1)))

Overwriting vehicle_detector.py
 vehicle_detector.py written(1187 bytes)


In [1]:
#This doesnt look awesome but seems to be functional

In [16]:
from IPython import get_ipython
#Going to attempt to load cloned repo weights from local source
code2=r"""
import yolov5
import numpy as np

#This .pt file is the one pulled from the wasdac9 repo listed above
#Since we are unable to call it using online methods we will do it this way
lp_weights = r"C:\Users\super\Models\anpr\best.pt"

#now load a yolo model using the lp_weights assigned above instead of trying to call the model as before
model_lp = yolov5.load(lp_weights)
#For initial pass of model will follow default values as shown in documentation
# https://huggingface.co/keremberke/yolov5n-license-plate
#Setting the NMS confidence threshold value
model_lp.conf = 0.25
#Setting the NMS IoU threshold value
model_lp.iou = 0.45

#Create a function that uses the model and weights above to extract license plates from new data when called
def detect_plates(img_bgr: np.ndarray):
    #img_bgr -- uint8 NumPy array in OpenCV format, can be converted to RGB if needed
    #returns a list of [x1,y1,x2,y2] plate boxes in absolute pixel coordinates
    results = model_lp(img_bgr, size=640)
    plates = [box[:4].cpu().numpy()
              for box in results.xyxy[0]]
    return plates
"""
get_ipython().run_cell_magic('writefile', 'lp_detector.py', code2)
#Ensure the file was written as expected 
print(" lp_detector.py written({} bytes)".format(len(code2)))

Overwriting lp_detector.py
 lp_detector.py written(1084 bytes)


In [58]:
#!!!!!!ATTENTION!!!!!
#  JUNK DO NOT UNCOMMENT
#  LEFT TO SHOW ORIGINAL IMPLEMENTATION OF EASYOCR
#  NOT USED IN PIPELINE

#Ok, so managed to get the first two layers assembled, with the vehicle recognition being the first
# and the license plate recognition being the second. Now we need to add a third layer to the stack which will
#enable us to extract the text from the license plate in the image and store it as plain text 
#To do this, since we are already using PyTorch, might as well use GPU accelerated services, 
# as such we have selected EasyOCR. In the future if other hardware is considered this stage may be written 
# using Tesseract instead of Easy OCR to enable functionality with CPU only systems or those otherwise 
#lacking discrete GPU(s).
#from IPython import get_ipython
#EasyOCR is installed using conda/bash so there is no pip install corresponding with it in the notebook
#code3=r"""
#import easyocr
#import cv2
#import numpy as np

#reader=easyocr.Reader(['en'])

#Create function to plug into pipeline for text extraction
#def plate_to_text(img_bgr: np.ndarray, mode:str="all"):
    #Since we are using img_bgr in the above layer we need to continue to use it for compatibility 
    #With easyocr we can take multiple passes of the same text and take the reading with the highest confidence
    #We first need to preprocess the image and make text extraction easier by enlarging the image 
    #and enhancing the contrast

    #Note: Text extraction on license plates is not accurate, in other words, 
    #it is not extracting the correct area of the tag and instead just pulling 
    #snipits. Will attempt two methods below which should resolve this issue. 
    #
    #Largest: Method will pull only the largest text from the license plate 
    #which should be the actual plate number
    #
    #All: For uses where the other information may be relevant, will extract 
    #all of the text from the license plate so should contain the state, county, 
    #and the plate number as well as any other text on the plate. 
    #
    #img = cv2.resize(img_bgr, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
    #gray= cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    #_, thr = cv2.threshold(gray, 0, 255, cv2.THRESH_OTSU + cv2.THRESH_BINARY)
    #thr_rgb = cv2.cvtColor(thr, cv2.COLOR_GRAY2RGB)
    
    #result = reader.readtext(thr_rgb, detail=1, paragraph=False)
    #Choose value with highest confidence
    #return max(result, key=len) if result else ''
    #No longer using above method due to not pulling plate number
    #if not results:
       # return ""

    #Define "Largest" mode of function 
   # if mode == "largest":
       # bbox, text, _ = max(results, key=lambda r: (abs((r[0][2][0]-r[0][0][0]*
                                                      #  (r[0][2][1]-r[0][0][1]))))
      #  return text.strip()

  #  elif mode == "all":
    #    results.sort(key=lambda r: min(pt[0] for pt in r[0]))  
     #   return " ".join(text.strip() for _, text, _ in results)

   # else:
   # raise ValueError("Mode must be set to 'Largest' or 'All' ")

#"""

                       
#get_ipython().run_cell_magic('writefile', 'lp_reader.py', code3)
#Ensure the file was written as expected 
#print(" lp_reader.py written({} bytes)".format(len(code3)))


In [54]:
#NOTE
#After several iterations of the license plate reader and continuing to build on each 
#iteration the versions have become both messy and complicated
#Additionally, it is very likely that many of the changes are unnecessary and as a result 
#should be removed from the code altogether. 
#Futher, according to source on EasyOCR, the default functionality should make it more than feasible to extract
#license plate text with higher accuracy than currently achieved 
from IPython import get_ipython

code_ocr_basic5="""
import easyocr
import cv2

reader = easyocr.Reader(['en'], gpu=True)

#Define basic function for pulling in output from the plate detector
def read_plates_basic1(img_bgr, plates,
                       #Add margin to image to allow for space for cropping 
                       #which may or may not help with the text recognition
                       margin: int=6,
                       #Define an allow list in an attempt to restrict using symbols as input
                       allowlist: str="0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"):
    h ,w = img_bgr.shape[:2]
    #Initialize a variable to store results in otherwise attempting iteration throws errors
    texts = []

    for idx, (x1, y1, x2, y2) in enumerate(plates, start=1):
        x1=max(int(x1)-margin,0)
        y1=max(int(y1)-margin,0)
        x2=min(int(x2)+margin,w-1)
        y2=min(int(y2)+margin,h-1)

        roi_bgr = img_bgr[y1:y2, x1:x2]
        #Must convert back to RGB as that is what OCR expects as input
        roi_rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB)
        
        result=reader.readtext(
            roi_rgb,
            detail=1, 
            paragraph=False, 
            allowlist=allowlist

        )
        #Threshold for text characters to be passed through
        min_prob = 0.30 
        #Passing in strings 
        candidates = [(t[1].strip(), t[2])       
                      for t in result if t[2] >= min_prob]

        if not candidates:
            texts.append("")                     
        else:
            #concatenate every candidate in reading order
            full = "".join(c[0] for c in candidates)
            texts.append(full)
    return texts

        #Not used left over from previous 
        #REMOVE ON REWRITE
        #for (bbox, text, prob) in result:
           # (top_left, top_right, bottom_right, bottom_left) = bbox
           # print(f'Text: {text}, Probability: {prob}')

           
           
      """

get_ipython().run_cell_magic('writefile', 'ocr_basic5.py', code_ocr_basic5)
#Ensure the file was written as expected 
print(" ocr_basic5.py written({} bytes)".format(len(code_ocr_basic5)))

Writing ocr_basic5.py
 ocr_basic5.py written(1906 bytes)


In [57]:
from IPython import get_ipython
#!!!ATTENTION
#PREVIOUS ITERATION
#NOT USED, OCR_BASIC5.PY IS BEING USED IN THE FIRST STAGE PIPELINE
#PREVIOUS ITERATION

code5="""

import easyocr
import cv2
import numpy as np


reader = easyocr.Reader(['en'])

def plate_to_text(img_bgr: np.ndarray, mode: str = "largest") -> str:

    # enlarge + threshold
    up   = cv2.resize(img_bgr, None, fx=2, fy=2,
                      interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(up, cv2.COLOR_BGR2GRAY)
    _, thr = cv2.threshold(gray, 0, 255,
                           cv2.THRESH_OTSU + cv2.THRESH_BINARY)
    thr_rgb = cv2.cvtColor(thr, cv2.COLOR_GRAY2RGB)

    # detailed OCR output  [bbox, text, conf]
    results = reader.readtext(thr_rgb, detail=1, paragraph=False)
    if not results:
        return ""

    #Create helper box instead of declaring key inside of largest option
    area = lambda r: abs((r[0][2][0] - r[0][0][0]) *
                         (r[0][2][1] - r[0][0][1]))
                         #This is identical to what was in largest but since we are adding another option to 
                         #try and increase accuracy might as well not run it twice
                         
    #Modify the "largest" mode to call the area instead of its own lambda function
    if mode == "largest":
        # pick entry whose bbox area is greatest
       return max(results, key=area)[1].strip()

    #Add another mode to takes in the 2nd and 3rd largest blocks as well to try and capture the entire license plate
    #instead of just a partial, as well as the State or County, not sure which it will pull. If this is succesfull 
    #may expand to 4th largest as well to try and capture either state or county, whichever is missing from this grab. 
    #Concerns still exist about the accuracy of the reader however pictures can always be included in tickets to avoid
    #errors based on this reader. 
    #Likewise, if this doesnt work will create a Tesseract layer and try again to see how it performs. 

    elif mode == "largest plus":
        #Largest lettering by area on the plate
        top3 = sorted(results, key=area, reverse=True)[:3]
        #Try to sort so the plate remains accurate
        top3.sort(key=lambda r: min(pt[0] for pt in r[0]))
        return " ".join(t.strip() for _, t, _ in top3)
        
    #Include an all option which just pulls all the text from the license plate
    elif mode == "all":
        # sort left to right by bbox x-min
        results.sort(key=lambda r: min(pt[0] for pt in r[0]))
        return " ".join(t.strip() for _, t, _ in results)

    else:
        raise ValueError("mode must be 'largest', 'largest plus' or 'all'")
"""
get_ipython().run_cell_magic('writefile', 'lp_reader3.py', code5)
#Ensure the file was written as expected 
print(" lp_reader3.py written({} bytes)".format(len(code5)))

Overwriting lp_reader3.py
 lp_reader3.py written(2522 bytes)


In [15]:
#!!!!!!!ATTENTION!!!!!!!!
#THIS IS NOT CALLED IN THE FIRST STAGE DOWN BELOW
#THIS IS A PREVIOUS ITERATION USED IN TESTING 
#MAKES USE OF TESSERACT INSTEAD OF EASCYOCR
#HOWEVER MADE SAME MISTAKES IN THIS CODE AS I DID FOR EASYOCR INITIALLY 
#PREVIOUS ITERATION
#NOT USED
#!!!!!!!!ATTENTION!!!!!!!!!!!

#Continuing to have issues with the accuracy when using EasyOCR for text extraction off of the license plates. 
#Will attempt to implement the same layer as above but using Tesseract instead. And it looks like that will need to 
#be done in the terminal so lets go fudge about in that for a minute.

#Tesseract installed via terminal, will try to do a test run on the cell to make sure the import call works, 
#if not may need to set the path manually 
#import pytesseract
#Ok, test didnt throw any errors so import seems successful. 
#Code should overall be the same regardless of method used so just going to copy code from cell above 
#and modify to pull tesseract instead. 
from IPython import get_ipython


code16="""
import pytesseract
import cv2
import numpy as np
#For some reason getting errors for the output, argument may not align with currently installed version of Tesseract

try:
    from pytesseract import Output
    _Out = Output.DICT
except ImportError:
    _Out = 'dict'
#Hopefully this will catch any errors and resolve them


#Upon further investigation there are more differences than I thought, will write from scratch
def tess_preprocess(bgr: np.ndarray):
    #Upscale as before using cv2 to resize
    crop = plate_crop
    crop = cv2.resize(crop, None, fx=8, fy=8, interpolation=cv2.INTER_CUBIC)
    gray = cv2.cvtColor(up, cv2.COLOR_BGR2GRAY)
    gray = cv2.createCLAHE(2.0, (8,8)).apply(gray)
    
    #Need to replace OTSU threshold with cv2.AdaptiveThreshold 
    #This is because OTSU is optimized for dark characters on light backgrounds, however the target plates
    #we are using (at least for the state of TN) the characters are white on a dark background.
    #Going to try to swap input and go back to using OTSU
    _,thr = cv2.threshold(
        gray, 0, 255, 
        cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        #If necessary invert the pixels to get a cleaner image
    if np.mean(thr) > 127:
        thr = cv2.bitwise_not(thr)

    thr = cv2.dilate(thr, np.ones((3,3), np.unit8),1)
    serial=pytesseract.image_to_string(thr, config=cfg).strip()
    return thr

#Adding individual config variable for plate reader to whitelist alphanumeric characters in attempt to 
#denoise the image or reduce the amount of noise the text reader is taking in as input
cfg= "--oem 1 --psm 8 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

def plate_to_text(img_bgr: np.ndarray, mode:str="all"):
    thr=tess_preprocess(img_bgr)
    data=pytesseract.image_to_data(
        thr, output_type=_Out,
        config=cfg
    )

    n = len(data["text"])
    boxes = []
    for i in range(n):
        txt = data["text"][i].strip()
        if not txt:                # skip blanks
            continue
        x, y, w, h = (data[k][i] for k in ("left","top","width","height"))
        boxes.append(((x, y, w, h), txt))

    if not boxes:
        return ""

    # area helper
    area = lambda b: b[0][2] * b[0][3]

    if mode == "largest":
        return max(boxes, key=area)[1]

    elif mode == "largest_plus":
        top3 = sorted(boxes, key=area, reverse=True)[:3]
        top3.sort(key=lambda b: b[0][0])           # left → right
        return " ".join(t for _, t in top3)

    elif mode == "all":
        boxes.sort(key=lambda b: b[0][0])
        return " ".join(t for _, t in boxes)

    else:
        raise ValueError("mode must be 'largest', 'largest_plus', or 'all'")
    

"""
get_ipython().run_cell_magic('writefile', 'lp_reader14.py', code16)
#Ensure the file was written as expected 
print(" lp_reader14.py written({} bytes)".format(len(code16)))


Writing lp_reader14.py
 lp_reader14.py written(2498 bytes)


In [5]:
#hmmmm. Seems the connection issues before were indeed on my end. 
# Read documenation on GitHub and found that I had failed to enable TCP port exceptions. 
# Ports are enabled and now able to make connections as expected. 

In [55]:
from IPython import get_ipython
#Bringing the layers together to work as a unit
code126=r"""
import cv2
#Call in the previously made pipelines
from vehicle_detector import detect_vehicle
from lp_detector import detect_plates
from ocr_basic5 import read_plates_basic1

#Create new function to pull images from input and determine if vehicle exist in image
#calling 1st layer
def process_image(path):
    img=cv2.imread(path)
    has_vehicle, veh_boxes = detect_vehicle(img)
    #If vehicle does not exist then we have no need for the next two layers
    #So we need to stop the processing and store nothing from this step
    if not has_vehicle:
        print("No Vehicle--Skipping Image")
        return None

    #If a vehicle does exist in the image we need to crop the vehicle region and narrow the search 
    # for the plate down to the expected area instead of re-analyzing the entire image again with the next layer
    for box in veh_boxes:
        x1,y1,x2,y2 = map(int, box)
        veh_crop = img[y1:y2, x1:x2]
        lp_boxes = detect_plates(veh_crop)

        #Now at this stage, if there is a license plate we want to pass it to the next layer to extract the text
        #from the license plate. 
        #for lp in lp_boxes:
           #lx1,ly1,lx2,ly2 = map(int, lp)
            #plate_crop = veh_crop[ly1:ly2,lx1:lx2]
            #text = read_plates_basic1(veh_crop)
            #print(f'Plate Number: {text}')

        #Need to rewrite for new dual input of the plate reader
        read_plates_basic1(veh_crop, lp_boxes)
"""

get_ipython().run_cell_magic('writefile', 'vehicle_pipeline25.py', code126)
#Ensure the file was written as expected 
print(" vehicle_pipeline25.py written({} bytes)".format(len(code126)))

Writing vehicle_pipeline25.py
 vehicle_pipeline25.py written(1450 bytes)


In [1]:
#Vehicle pipeline completed, now need to test with some images 
#however a quick look at kaggle either shows vehicles with tags blurred 
# or pictures of tags with no vehicle. Some exist with vehicles and tags 
# but are in a non-roman alphabet which may present issue with the model 
# and is outside the scope of the project
#As a result will try to obtain some level of test photos to use for an 
# initial test run before collecting video to try. 

In [56]:
#Ok, pictures are obtained, a couple of notes. 
#the pictures taken are in 4k resolution, (3072x4080), but YOLOv5
#scales the picture internally so declaring resolution changes isnt required
#Pictures are saved to the desktop in a folder, so will attempt to iterate through
#the folder and use the pipeline on each picture in the folder. 
import pathlib
import importlib
import cv2
import numpy as np
#This is the wrapper that contains the process from above(the three models bundeled)
from vehicle_pipeline25 import process_image
#We need to import these again despite importing process_image because we want to 
#perform additional QA on the output so want to see what the model is looking at
#To accomplish this we will need to run these models again in brief
from vehicle_detector import detect_vehicle 
from lp_detector import detect_plates
import ocr_basic5
#importlib.reload(lp_reader2)
from ocr_basic5 import read_plates_basic1

#Next we need to configure where the test images are being stored, and then where
#we want the model to save the output. 
#Where the pictures are being pulled from 
img_dir = r"C:\Users\super\OneDrive\Desktop\car photos"
#Where we want to save the annotated versions, this should be in the same folder as original
save_dir = pathlib.Path(img_dir)/"results_ocr_basic_jun18"
show_each=False
save_dir.mkdir(exist_ok=True)

#Declare the input type expected, since they are pictures we will use photo formats
img_paths = sorted(p for p in pathlib.Path(img_dir).glob("*")
                   if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp"})

#Begin iteration process, if image is not able to be called throw error
for p in img_paths:
    img=cv2.imread(str(p))
    if img is None:
        print(f"{p.name:30} Could Not Read File")
        continue

    #Run the pipeline on the image
    process_image(str(p))

    #Call detect vehicle to draw boxes if vehicle is detected for later inspection
    has_vehicle, veh_boxes = detect_vehicle(img)
    #If their is no vehicle skip the image and go to the next one
    if not has_vehicle:
        continue

    #If there is a vehicle then we want to draw the bounding boxes
    for vx1,vy1,vx2,vy2 in map(lambda b: map(int, b), veh_boxes):
        cv2.rectangle(img, (vx1,vy1), (vx2, vy2), (0,255,0), 2)
        veh_crop = img[vy1:vy2, vx1:vx2]


        #Same for the license plates. If there is a car then we want to examine the license 
        #plate. Here we will draw license plate bounding boxes in the image to make sure
        #they are bounding the correct thing
        # Detect plates inside the vehicle crop
        lp_boxes = detect_plates(veh_crop)                 # list of [x1,y1,x2,y2]

        # OCR every plate in one shot (read_plates_basic1 must *return* text per box)
        lp_texts = read_plates_basic1(veh_crop, lp_boxes)  # ← now 2 args

        # Draw boxes & labels
        for (lx1, ly1, lx2, ly2), text in zip(lp_boxes, lp_texts):
            lx1, ly1, lx2, ly2 = map(int, (lx1, ly1, lx2, ly2))
            abs_box = (vx1 + lx1, vy1 + ly1, vx1 + lx2, vy1 + ly2)

            cv2.rectangle(img, (abs_box[0], abs_box[1]),
                                (abs_box[2], abs_box[3]), (255, 0, 0), 2)

            cv2.putText(img, text or "?", (abs_box[0], abs_box[1] - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2,
                        cv2.LINE_AA)

            print(f"{p.name:30} {text or '(no OCR)'}")
        


    if show_each:
        cv2.imshow("result", img)
        cv2.waitKey(0)

    cv2.imwrite(str(save_dir/p.name),img)

cv2.destroyAllWindows()

print(f"Process Completed, Images Process. All annotated images saved to {save_dir}")

#Going to run the cell once without saving the cell as .py to see if functional

#First pass resulted in low performance in tag text extraction. Text was extracted
#from the tag but it was not the tag number. Have edited the ocr function, will 
#perform second test of function. 

#Second pass results were slightly better but some of the text is still inaccurate
#Part of the issue may be small text in the bounding box interfering with output. 
#Will restrict text extraction to largest characters to see if accuracy improves
#and run the pipeline again
#
#Third pass resulted in an interesting outcome where only one half of the license 
# plate was read, and even reading half the accuracy was still low. 
# Will need to conitnue to iterate on this and increase functionality. 
#Not in an acceptable state as is. May need to select different weights
#or maybe even a different model altogether. The license plate identification 
#is working just fine but the text extraction layer is exhibiting abysmal performance. 

C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-package

PXL_20250511_190945899.jpg     LUNTDMGTENNESSEE10425820NVACATIOMBHULLSULLIVAN


C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-package

PXL_20250511_190946955.jpg     THETENNESSEE10251820NVACATIOBHULLSULLIVAN


C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-package

...

C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-pa

PXL_20250529_202422256.jpg     (no OCR)
PXL_20250529_202422256.jpg     VA0J92628VA681781APRVIRGINIA24TRC4664VIRGINIAISFORLOOERSVO09


C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\si

PXL_20250529_202437764.jpg     (no OCR)
PXL_20250529_202437764.jpg     6791182MAYVIRGINIA25THYF4208VIRGINIAISFORLOOERSVI09
Process Completed, Images Process. All annotated images saved to C:\Users\super\OneDrive\Desktop\car photos\results_ocr_basic_jun18


C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super/.cache\torch\hub\ultralytics_yolov5_master\models\common.py:906: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-packages\yolov5\models\common.py:709: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast(autocast):
C:\Users\super\anaconda3\envs\torch39\lib\site-pa

In [29]:
#Pipeline is currently functional but need to validate each of the 
#pipes to ensure accuracy at each stage. Does little good for the pipe to
#be functional if the pipe is only acheiving 30% accuracy. Since transfer learning
#is being utilized the accuracy should be high "out of the box" so there may 
#not be much if any fine tuning needed. 
#However, as seen above, the license plate reader is exhibiting low performance
#This may result in dropping EasyOCR and replacing it with Tesseract. 


#After making multiple versions of each will need to go back through and see which one actually shows the most 
#promise. Neither one of them are standing out as especially performant, more research will need to be conducted
#on the matter. For now need to move to next pipeline and finish off coding draft. 
#The project wont be a perfect iteration of the idea but it should be funtional all the same.

#Turns out I wasnt passing all the required input which caused significant degradation in the performance 
#of the text extraction using both EasyOCR and Tesseract methods. Scrapping the entire pipe and rewriting it 
#from scratch was the answer, it contained too much bloat trying to resolve an issue I didnt have while 
#ignoring the one I did. 

#With the rewritten EasyOCR the text extraction pulls text thats much much more accurate from every picture 
#where the license plate is recognized. This shows substantial improvement over previous iterations, and 
#at this point feel comfortable enough to start expanding the dataset versus tuning the pipes



In [1]:
#pip install ultralytics==8.3.130

In [6]:
#import mediapipe                      # top-level package
#from mediapipe import solutions as mp # Hands, Pose, etc.

#print(mediapipe.__version__)          # e.g. 0.10.8

0.10.8


In [7]:
# human_detector.py
from IPython import get_ipython

code200="""
from ultralytics import YOLO
import numpy as np

_PERSON_CLS = 0                       # COCO index for “person”
model = YOLO('yolov8n.pt')            # 3 MB, fast

def detect_people(img_bgr: np.ndarray, return_boxes=False):
    #True/False (and optional boxes) indicating at least one person
    res   = model(img_bgr, imgsz=640, verbose=False)
    boxes = [b.xyxy[0].cpu().numpy()
             for b in res[0].boxes
             if int(b.cls) == _PERSON_CLS and b.conf > 0.25]
    return (bool(boxes), boxes) if return_boxes else bool(boxes)

"""
get_ipython().run_cell_magic('writefile', 'hugh_man_detector.py', code200)
#Ensure the file was written as expected 
print("hugh_man_detector.py written({} bytes)".format(len(code200)))

Writing hugh_man_detector.py
hugh_man_detector.py written(546 bytes)


In [13]:
from IPython import get_ipython

code300="""
# hand_object_detector.py
from mediapipe import solutions as mp
from ultralytics import YOLO
import cv2, numpy as np, contextlib

# Load once
hands = mp.hands.Hands(static_image_mode=True,
                       max_num_hands=2,
                       min_detection_confidence=0.4)
obj_model = YOLO('yolov8s.pt')       
_PERSON = next(k for k, v in obj_model.names.items() if v == "person")

@contextlib.contextmanager
def _mp_context():
    try:
        yield
    finally:
        hands.close()

def _hand_rois(img_bgr, person_boxes, pad=25):
    H,W,_ = img_bgr.shape
    rois = []
    for x1,y1,x2,y2 in person_boxes:
        crop = img_bgr[int(y1):int(y2), int(x1):int(x2)]
        rgb  = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        res  = hands.process(rgb)
        if not res.multi_hand_landmarks:
            continue
        for lm in res.multi_hand_landmarks:
            xs = [l.x for l in lm.landmark]; ys = [l.y for l in lm.landmark]
            hx1 = max(int(min(xs)*crop.shape[1]) - pad, 0)
            hy1 = max(int(min(ys)*crop.shape[0]) - pad, 0)
            hx2 = min(int(max(xs)*crop.shape[1]) + pad, crop.shape[1])
            hy2 = min(int(max(ys)*crop.shape[0]) + pad, crop.shape[0])
            # translate to full-image coords
            rois.append((x1+hx1, y1+hy1, x1+hx2, y1+hy2))
    return rois


#Function for object identification in each hand
def objects_in_hands(img_bgr: np.ndarray, person_boxes):
    has_obj, details = False, []
    rois = _hand_rois(img_bgr, person_boxes)
    for x1,y1,x2,y2 in rois:
        patch = img_bgr[int(y1):int(y2), int(x1):int(x2)]
        if patch.size == 0:
            continue
        # upscale small hands for clarity
        if patch.shape[0] < 80:
            patch = cv2.resize(patch, None, fx=3, fy=3,
                               interpolation=cv2.INTER_CUBIC)

        res = obj_model(patch, imgsz=320, verbose=False)
        for b in res[0].boxes:
            cls = int(b.cls); conf = float(b.conf)
            if cls == _PERSON or conf < 0.3:
                continue
            has_obj = True
            box = b.xyxy[0].cpu().numpy()
            # scale back & translate to full image
            scale = patch.shape[0] / (y2 - y1)
            box /= scale
            box[[0,2]] += x1; box[[1,3]] += y1
            details.append((obj_model.names[cls], conf, box))
    return has_obj, details

"""

get_ipython().run_cell_magic('writefile', 'hand_object_detector.py', code300)
#Ensure the file was written as expected 
print("hand_object_detector.py written({} bytes)".format(len(code300)))

Overwriting hand_object_detector.py
hand_object_detector.py written(2415 bytes)


In [17]:
# people_pipeline.py
from IPython import get_ipython

code900="""
import numpy as np
import cv2
from hugh_man_detector       import detect_people
from hand_object_detector import objects_in_hands

def process_frame(path_or_bgr):
    img = (cv2.imread(path_or_bgr) if isinstance(path_or_bgr,str)
           else path_or_bgr)
    if img is None:
        raise ValueError("cannot open image")

    has_pers, pers_boxes = detect_people(img, return_boxes=True)
    if not has_pers:
        return {"person": False, "object": False}

    has_obj, obj_list = objects_in_hands(img, pers_boxes)
    return {"person": True, "object": has_obj, "details": obj_list}
"""
get_ipython().run_cell_magic('writefile', 'people_pipeline.py', code900)
#Ensure the file was written as expected 
print("people_pipeline.py written({} bytes)".format(len(code900)))

Overwriting people_pipeline.py
people_pipeline.py written(589 bytes)


In [19]:
import json, pathlib, cv2
from people_pipeline import process_frame
import numpy as np

IMG_DIR  = pathlib.Path(r"C:\Users\super\OneDrive\Desktop\People in Cars")
OUT_JSON = IMG_DIR / "people_results.json"

#Initialize results variable for final output
results = []

for img_path in sorted(IMG_DIR.glob("*")):
    if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png", ".bmp"}:
        continue

    out = process_frame(str(img_path))
    #retain file name for refernce to original image
    out["file"] = img_path.name
    #Initialize variable for details output from each picture/frame
    clean_details = []
    for label, conf, box in out.get("details", []):
        if isinstance(box, np.ndarray):
            box = box.tolist()
        #cast the confidence to float for cleaner output for probabilities
        clean_details.append([label, float(conf), box])

    out["details"] = clean_details
    results.append(out)
    #output the results by listing boolean true/false value for person and object
    #output should also contain the label for the object in the picture
    print(f"{img_path.name:30} | "
          f"person: {out['person']} | object: {out['object']}")

# write once after the loop to JSON
with OUT_JSON.open("w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"\n✓  Saved {len(results)} results → {OUT_JSON}")

test001_np.png                 | person: True | object: False
test002_np.png                 | person: True | object: False
test003_np.png                 | person: True | object: False
test004_np.png                 | person: True | object: False
test005_np.png                 | person: True | object: False
test006_np.png                 | person: True | object: False
test007_np.png                 | person: True | object: False
test008_rand.png               | person: True | object: True
test009_np.png                 | person: True | object: False
test010_p.png                  | person: True | object: True
test011_p.png                  | person: True | object: False
test012_p.png                  | person: True | object: False
test013_p.png                  | person: True | object: True
test014_p.png                  | person: True | object: True
test015_p.png                  | person: True | object: True
test016_p.png                  | person: True | object: True
test017_p.png 